This notebook will be used to extract information from the point clouds, and associated files, so that we may use it in a model to understand how the surveys and optimal interpolation methods interact across various geographical regions and surveying regimes.

Some characteristics that will probably be beneficial:
1. Depth Coefficient of variation
2. Depth or survey point anisotropy (e.g. anisotropy ratio of depth measurements or of survey point density/location)
3. Point density

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../../src'))

from pathlib import Path
import processing_help
import warnings
import re
import pandas as pd
import geopandas as gpd
import fiona
import numpy as np
from sklearn.preprocessing import StandardScaler

# you can suppress some warnings if you wish. Some libraries further in the processing pipeline will benefit from this.
warnings.filterwarnings('ignore')

# Directly for downloaded data from eHydro

In [ ]:
# DATA_DIR = Path(os.getcwd()).parent.parent / 'raw_data'

DATA_DIR = Path('/mnt/teamgroup/AutoDBM/source_data')
os.listdir(DATA_DIR)

In [ ]:
survey_gdbs = list(DATA_DIR.glob('*/*/*.gdb'))

In [ ]:
survey_characteristics = {}

for file in survey_gdbs:
    if 'SurveyPoint_HD' in fiona.listlayers(file):

        data = gpd.read_file(file, layer='SurveyPoint_HD')
        if len(data) < 10000:
            print(f"Point cloud for {file.stem} is less thank 10,000 points.")
            continue
        else:
            data_x = data.xLocation.values
            data_y = data.yLocation.values
            data_z = data.Z_use.values

            # TODO: maybe look into the impact of rounding these numbers. Obviously we lose a bit of information, but also saves storage space
            # and it may reduce the compute time need for GLMM + EMM if we use float32 instead of float64, for example

            survey_characteristics[file.stem] = processing_help.calculate_survey_characteristics(data_x, data_y, data_z)
    else:
        print(f'SurveyPoint_HD not available for {file}')

In [ ]:
survey_characteristics_df = (
    pd.DataFrame.from_dict(
        survey_characteristics,
        orient="index",
        columns=["density", "depth_cv", "anisotropy"],
    )
    .reset_index()
    .rename(columns={"index": "survey"})
)

survey_characteristics_df["survey"] = survey_characteristics_df["survey"].astype(str)
survey_characteristics_df.head()

In [ ]:
# select some of the more diverse surveys based on the three characteristics

# TODO: A figure showing your 100 selected surveys in the 3D space of density × depth CV × anisotropy with all surveys in the back

features = survey_characteristics_df[["density", "depth_cv", "anisotropy"]].values
features_scaled = StandardScaler().fit_transform(features)

# Greedy MaxMin selection
def select_diverse(X, n=100):
    n_samples = len(X)
    selected = [np.random.randint(n_samples)]  # random seed point
    min_dists = np.full(n_samples, np.inf)

    for _ in range(n - 1):
        last = X[selected[-1]]
        dists = np.linalg.norm(X - last, axis=1)
        min_dists = np.minimum(min_dists, dists)
        min_dists[selected] = -1  # exclude already chosen
        selected.append(np.argmax(min_dists))

    return selected

indices = select_diverse(features_scaled, n=100)
diverse_subset = survey_characteristics_df.iloc[indices]
print(len(diverse_subset))

In [ ]:
diverse_subset.to_csv(Path(os.getcwd()).parent.parent / 'analysis_data' / 'survey_characteristics.csv', index=False)

# Some brainstorming on how to get data to R for GLMM + EMM

- Each generated surface has a corresponding '_validation.csv' which gives the XY position, observed depth, predicted depth, and residual.
- One cumulative file for the survey characteristics

Probably read them all in and connect by survey? Would need to calculate the mean RMSE for each survey using the data available in the indiviudal _validation.csv files